# Tái lập huấn luyện BTS trên NYU Depth V2


## 1. Thiết kế thực nghiệm

| Thành phần | Thiết lập |
|---|---|
| Model | BTS |
| Encoder | DenseNet161, ImageNet pretrained |
| Dataset | NYU Depth V2 |
| Training samples | 24,231 |
| Test samples | 654 |
| Input training | 416 × 544 |
| Global batch size | 4 |
| GPU | 2 × Tesla T4 |
| Epochs | 50 |
| Optimizer | AdamW |
| Initial learning rate | 1e-4 |
| Weight decay | 1e-2 |
| Adam epsilon | 1e-3 |
| Depth range | 1e-3 m – 10 m |
| Data augmentation | Random rotation ±2.5° |
| Online evaluation | Mỗi 500 global steps |
| Evaluation crop | Eigen crop |
| Training loss | Scale-Invariant Logarithmic Loss (SILog) |

### Kiểm thử tiền nghiệm đã hoàn tất

Trước full run, pipeline đã được kiểm tra trên cùng môi trường 2 × T4:

| Kiểm thử | Kết quả |
|---|---|
| DDP training 416×544, global batch 4 | 10/10 steps hoàn thành |
| Rolling recovery checkpoint | `model-latest` hợp lệ |
| Mid-epoch resume | checkpoint step 8 → tiếp tục đúng tại step 9 |
| Optimizer state khi resume | khôi phục thành công |
| Online evaluation | đủ 654 mẫu |
| 9 depth metrics | tính toán và đồng bộ giữa 2 GPU |
| Best-checkpoint tracking | hoạt động |




In [ ]:
import os
from pathlib import Path

TEMP = Path('/kaggle/temp')
WORK = Path('/kaggle/working')
MODEL_NAME = 'bts_nyu_kaggle_full'

RUN_TRAINING = True
RUN_FINAL_EVALUATION = True

# WandB Real-time Dashboard Configuration
os.environ['WANDB_API_KEY'] = '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'
os.environ['WANDB_PROJECT'] = 'bts-nyuv2-depth-research'
os.environ['WANDB_ENTITY'] = 'edward-carlos731-industrial-university-of-ho-chi-minh-city'

# 1. Dynamic Auto-Detection for NYUv2 dataset via unique pretrained weight or zip file
INPUT = None
input_base = Path('/kaggle/input')
if input_base.exists():
    for p in input_base.rglob('densenet161-8d451a50.pth'):
        INPUT = p.parent
        break
    if INPUT is None:
        for p in input_base.rglob('*bts_kaggle_source*'):
            INPUT = p.parent
            break

if INPUT is None:
    INPUT = Path('/kaggle/input/bts-nyuv2-full-training')

# 2. Dynamic Auto-Detection for model-latest checkpoint
RESUME_CHECKPOINT = ''
if input_base.exists():
    for ckpt_p in input_base.rglob('model-latest'):
        if ckpt_p.is_file() and ckpt_p.stat().st_size > 100 * 1024**2:
            RESUME_CHECKPOINT = str(ckpt_p)
            print(f'[AUTO-RESUME] Phat hien checkpoint noi tiep tai: {ckpt_p}')
            break

if not RESUME_CHECKPOINT:
    working_latest = WORK / 'models' / MODEL_NAME / 'model-latest'
    if working_latest.exists():
        RESUME_CHECKPOINT = str(working_latest)
        print(f'[AUTO-RESUME] Phat hien checkpoint trong working: {working_latest}')

TRAIN_EPOCHS = 50
GLOBAL_BATCH_SIZE = 4
INPUT_HEIGHT = 416
INPUT_WIDTH = 544
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-2
ADAM_EPS = 1e-3
MAX_DEPTH = 10.0
EVAL_FREQ = 500
SAVE_FREQ = 500

print('Model:', MODEL_NAME)
print('Input dataset:', INPUT)
print('Resume checkpoint:', RESUME_CHECKPOINT or 'fresh run')
print('WandB Dashboard: https://wandb.ai/edward-carlos731-industrial-university-of-ho-chi-minh-city/bts-nyuv2-depth-research')


## 3. Môi trường thực thi

Phần này ghi nhận phiên bản Python, PyTorch, CUDA và GPU thực tế của phiên Kaggle. Thông tin được lưu lại trong manifest ở cuối notebook để bảo đảm khả năng truy vết thí nghiệm.


In [ ]:
import sys
import subprocess
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

assert torch.cuda.is_available(), "CUDA is not available."
assert torch.cuda.device_count() == 2, (
    "Notebook này được thiết kế cho Kaggle GPU T4 x2."
)

print()
print(subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True,
    check=True,
).stdout)


## 4. Chuẩn bị mã nguồn và dataset

Dataset Kaggle gồm bốn thành phần:

- mã nguồn BTS đã kiểm thử;
- NYUv2 synchronized training set;
- official test split;
- DenseNet161 ImageNet weights.

Dataset chỉ được đọc từ `/kaggle/input`. File tạm được đặt trong `/kaggle/temp`, còn checkpoint, log và kết quả cuối được lưu tại `/kaggle/working`.


In [ ]:
import os
import shutil
import zipfile
import hashlib
from pathlib import Path

assert INPUT.exists(), f"Không tìm thấy dataset: {INPUT}"

BTS_ROOT = TEMP / "bts"
NYU_ROOT = TEMP / "dataset" / "nyu_depth_v2"

def remove_runtime_path(path: Path):
    if path.is_symlink():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)

remove_runtime_path(BTS_ROOT)
remove_runtime_path(NYU_ROOT)

BTS_ROOT.mkdir(parents=True, exist_ok=True)
NYU_ROOT.mkdir(parents=True, exist_ok=True)

source_zip = INPUT / "bts_kaggle_source.zip"
source_dir = INPUT / "bts_kaggle_source"

if source_zip.exists():
    with zipfile.ZipFile(source_zip) as z:
        z.extractall(BTS_ROOT)
elif source_dir.exists():
    shutil.copytree(source_dir, BTS_ROOT, dirs_exist_ok=True)
else:
    raise FileNotFoundError("Không tìm thấy BTS source.")

train_root = NYU_ROOT / "sync"
sync_zip = INPUT / "sync.zip"
nested_sync = INPUT / "sync" / "sync"
flat_sync = INPUT / "sync"

if sync_zip.exists():
    with zipfile.ZipFile(sync_zip) as z:
        z.extractall(NYU_ROOT)
elif nested_sync.exists():
    train_root.symlink_to(nested_sync, target_is_directory=True)
elif flat_sync.exists():
    train_root.symlink_to(flat_sync, target_is_directory=True)
else:
    raise FileNotFoundError("Không tìm thấy NYUv2 sync dataset.")

official_root = NYU_ROOT / "official_splits"
official_root.mkdir(parents=True, exist_ok=True)
test_root = official_root / "test"

test_zip = INPUT / "official_splits_test.zip"
expanded_test = INPUT / "official_splits_test" / "test"

if test_zip.exists():
    with zipfile.ZipFile(test_zip) as z:
        z.extractall(official_root)
elif expanded_test.exists():
    test_root.symlink_to(expanded_test, target_is_directory=True)
else:
    raise FileNotFoundError("Không tìm thấy official NYUv2 test split.")

def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

print("BTS source:", BTS_ROOT)
print("Train root:", train_root)
print("Test root :", test_root)

# Inject WandB live tracking bts_main.py
(BTS_ROOT / 'pytorch' / 'bts_main.py').write_text('# Copyright (C) 2019 Jin Han Lee\n#\n# This file is a part of BTS.\n# This program is free software: you can redistribute it and/or modify\n# it under the terms of the GNU General Public License as published by\n# the Free Software Foundation, either version 3 of the License, or\n# (at your option) any later version.\n#\n# This program is distributed in the hope that it will be useful,\n# but WITHOUT ANY WARRANTY; without even the implied warranty of\n# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the\n# GNU General Public License for more details.\n#\n# You should have received a copy of the GNU General Public License\n# along with this program. If not, see <http://www.gnu.org/licenses/>\n\nimport time\nimport argparse\nimport datetime\nimport sys\nimport os\nimport shutil\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.utils as utils\n\nimport torch.backends.cudnn as cudnn\nimport torch.distributed as dist\nimport torch.multiprocessing as mp\n\nfrom tensorboardX import SummaryWriter\n\nimport matplotlib\nimport matplotlib.cm\nimport threading\nfrom tqdm import tqdm\n\nfrom bts import BtsModel\nfrom bts_dataloader import *\n\n\ndef convert_arg_line_to_args(arg_line):\n    for arg in arg_line.split():\n        if not arg.strip():\n            continue\n        yield arg\n\n\nparser = argparse.ArgumentParser(description=\'BTS PyTorch implementation.\', fromfile_prefix_chars=\'@\')\nparser.convert_arg_line_to_args = convert_arg_line_to_args\n\nparser.add_argument(\'--mode\',                      type=str,   help=\'train or test\', default=\'train\')\nparser.add_argument(\'--model_name\',                type=str,   help=\'model name\', default=\'bts_eigen_v2\')\nparser.add_argument(\'--encoder\',                   type=str,   help=\'type of encoder, desenet121_bts, densenet161_bts, \'\n                                                                    \'resnet101_bts, resnet50_bts, resnext50_bts or resnext101_bts\',\n                                                               default=\'densenet161_bts\')\n# Dataset\nparser.add_argument(\'--dataset\',                   type=str,   help=\'dataset to train on, kitti or nyu\', default=\'nyu\')\nparser.add_argument(\'--data_path\',                 type=str,   help=\'path to the data\', required=True)\nparser.add_argument(\'--gt_path\',                   type=str,   help=\'path to the groundtruth data\', required=True)\nparser.add_argument(\'--filenames_file\',            type=str,   help=\'path to the filenames text file\', required=True)\nparser.add_argument(\'--input_height\',              type=int,   help=\'input height\', default=480)\nparser.add_argument(\'--input_width\',               type=int,   help=\'input width\',  default=640)\nparser.add_argument(\'--max_depth\',                 type=float, help=\'maximum depth in estimation\', default=10)\n\n# Log and save\nparser.add_argument(\'--log_directory\',             type=str,   help=\'directory to save checkpoints and summaries\', default=\'\')\nparser.add_argument(\'--checkpoint_path\',           type=str,   help=\'path to a checkpoint to load\', default=\'\')\nparser.add_argument(\'--log_freq\',                  type=int,   help=\'Logging frequency in global steps\', default=100)\nparser.add_argument(\'--save_freq\',                 type=int,   help=\'Checkpoint saving frequency in global steps\', default=500)\n\n# Training\nparser.add_argument(\'--fix_first_conv_blocks\',                 help=\'if set, will fix the first two conv blocks\', action=\'store_true\')\nparser.add_argument(\'--fix_first_conv_block\',                  help=\'if set, will fix the first conv block\', action=\'store_true\')\nparser.add_argument(\'--bn_no_track_stats\',                     help=\'if set, will not track running stats in batch norm layers\', action=\'store_true\')\nparser.add_argument(\'--weight_decay\',              type=float, help=\'weight decay factor for optimization\', default=1e-2)\nparser.add_argument(\'--bts_size\',                  type=int,   help=\'initial num_filters in bts\', default=512)\nparser.add_argument(\'--retrain\',                               help=\'if used with checkpoint_path, will restart training from step zero\', action=\'store_true\')\nparser.add_argument(\'--adam_eps\',                  type=float, help=\'epsilon in Adam optimizer\', default=1e-6)\nparser.add_argument(\'--batch_size\',                type=int,   help=\'batch size\', default=4)\nparser.add_argument(\'--num_epochs\',                type=int,   help=\'number of epochs\', default=50)\nparser.add_argument(\'--learning_rate\',             type=float, help=\'initial learning rate\', default=1e-4)\nparser.add_argument(\'--end_learning_rate\',         type=float, help=\'end learning rate\', default=-1)\nparser.add_argument(\'--variance_focus\',            type=float, help=\'lambda in paper: [0, 1], higher value more focus on minimizing variance of error\', default=0.85)\n\n# Preprocessing\nparser.add_argument(\'--do_random_rotate\',                      help=\'if set, will perform random rotation for augmentation\', action=\'store_true\')\nparser.add_argument(\'--degree\',                    type=float, help=\'random rotation maximum degree\', default=2.5)\nparser.add_argument(\'--do_kb_crop\',                            help=\'if set, crop input images as kitti benchmark images\', action=\'store_true\')\nparser.add_argument(\'--use_right\',                             help=\'if set, will randomly use right images when train on KITTI\', action=\'store_true\')\n\n# Multi-gpu training\nparser.add_argument(\'--num_threads\',               type=int,   help=\'number of threads to use for data loading\', default=1)\nparser.add_argument(\'--world_size\',                type=int,   help=\'number of nodes for distributed training\', default=1)\nparser.add_argument(\'--rank\',                      type=int,   help=\'node rank for distributed training\', default=0)\nparser.add_argument(\'--dist_url\',                  type=str,   help=\'url used to set up distributed training\', default=\'tcp://127.0.0.1:1234\')\nparser.add_argument(\'--dist_backend\',              type=str,   help=\'distributed backend\', default=\'nccl\')\nparser.add_argument(\'--gpu\',                       type=int,   help=\'GPU id to use.\', default=None)\nparser.add_argument(\'--multiprocessing_distributed\',           help=\'Use multi-processing distributed training to launch \'\n                                                                    \'N processes per node, which has N GPUs. This is the \'\n                                                                    \'fastest way to use PyTorch for either single node or \'\n                                                                    \'multi node data parallel training\', action=\'store_true\',)\n# Online eval\nparser.add_argument(\'--do_online_eval\',                        help=\'if set, perform online eval in every eval_freq steps\', action=\'store_true\')\nparser.add_argument(\'--data_path_eval\',            type=str,   help=\'path to the data for online evaluation\', required=False)\nparser.add_argument(\'--gt_path_eval\',              type=str,   help=\'path to the groundtruth data for online evaluation\', required=False)\nparser.add_argument(\'--filenames_file_eval\',       type=str,   help=\'path to the filenames text file for online evaluation\', required=False)\nparser.add_argument(\'--min_depth_eval\',            type=float, help=\'minimum depth for evaluation\', default=1e-3)\nparser.add_argument(\'--max_depth_eval\',            type=float, help=\'maximum depth for evaluation\', default=80)\nparser.add_argument(\'--eigen_crop\',                            help=\'if set, crops according to Eigen NIPS14\', action=\'store_true\')\nparser.add_argument(\'--garg_crop\',                             help=\'if set, crops according to Garg  ECCV16\', action=\'store_true\')\nparser.add_argument(\'--eval_freq\',                 type=int,   help=\'Online evaluation frequency in global steps\', default=500)\nparser.add_argument(\'--eval_summary_directory\',    type=str,   help=\'output directory for eval summary,\'\n                                                                    \'if empty outputs to checkpoint folder\', default=\'\')\n\nif sys.argv.__len__() == 2:\n    arg_filename_with_prefix = \'@\' + sys.argv[1]\n    args = parser.parse_args([arg_filename_with_prefix])\nelse:\n    args = parser.parse_args()\n\nif args.mode == \'train\' and not args.checkpoint_path:\n    from bts import *\n\nelif args.mode == \'train\' and args.checkpoint_path:\n    model_dir = os.path.dirname(args.checkpoint_path)\n    model_name = os.path.basename(model_dir)\n    import sys\n    sys.path.append(model_dir)\n    for key, val in vars(__import__(model_name)).items():\n        if key.startswith(\'__\') and key.endswith(\'__\'):\n            continue\n        vars()[key] = val\n\n\ninv_normalize = transforms.Normalize(\n    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],\n    std=[1/0.229, 1/0.224, 1/0.225]\n)\n\neval_metrics = [\'silog\', \'abs_rel\', \'log10\', \'rms\', \'sq_rel\', \'log_rms\', \'d1\', \'d2\', \'d3\']\n\n\ndef compute_errors(gt, pred):\n    thresh = np.maximum((gt / pred), (pred / gt))\n    d1 = (thresh < 1.25).mean()\n    d2 = (thresh < 1.25 ** 2).mean()\n    d3 = (thresh < 1.25 ** 3).mean()\n\n    rms = (gt - pred) ** 2\n    rms = np.sqrt(rms.mean())\n\n    log_rms = (np.log(gt) - np.log(pred)) ** 2\n    log_rms = np.sqrt(log_rms.mean())\n\n    abs_rel = np.mean(np.abs(gt - pred) / gt)\n    sq_rel = np.mean(((gt - pred) ** 2) / gt)\n\n    err = np.log(pred) - np.log(gt)\n    silog = np.sqrt(np.mean(err ** 2) - np.mean(err) ** 2) * 100\n\n    err = np.abs(np.log10(pred) - np.log10(gt))\n    log10 = np.mean(err)\n\n    return [silog, abs_rel, log10, rms, sq_rel, log_rms, d1, d2, d3]\n\n\ndef block_print():\n    sys.stdout = open(os.devnull, \'w\')\n\n\ndef enable_print():\n    sys.stdout = sys.__stdout__\n\n\ndef get_num_lines(file_path):\n    f = open(file_path, \'r\')\n    lines = f.readlines()\n    f.close()\n    return len(lines)\n\n\ndef colorize(value, vmin=None, vmax=None, cmap=\'Greys\'):\n    value = value.cpu().numpy()[:, :, :]\n    value = np.log10(value)\n\n    vmin = value.min() if vmin is None else vmin\n    vmax = value.max() if vmax is None else vmax\n\n    if vmin != vmax:\n        value = (value - vmin) / (vmax - vmin)\n    else:\n        value = value*0.\n\n    cmapper = matplotlib.cm.get_cmap(cmap)\n    value = cmapper(value, bytes=True)\n\n    img = value[:, :, :3]\n\n    return img.transpose((2, 0, 1))\n\n\ndef normalize_result(value, vmin=None, vmax=None):\n    value = value.cpu().numpy()[0, :, :]\n\n    vmin = value.min() if vmin is None else vmin\n    vmax = value.max() if vmax is None else vmax\n\n    if vmin != vmax:\n        value = (value - vmin) / (vmax - vmin)\n    else:\n        value = value * 0.\n\n    return np.expand_dims(value, 0)\n\n\ndef set_misc(model):\n    if args.bn_no_track_stats:\n        print("Disabling tracking running stats in batch norm layers")\n        model.apply(bn_init_as_tf)\n\n    if args.fix_first_conv_blocks:\n        if \'resne\' in args.encoder:\n            fixing_layers = [\'base_model.conv1\', \'base_model.layer1.0\', \'base_model.layer1.1\', \'.bn\']\n        else:\n            fixing_layers = [\'conv0\', \'denseblock1.denselayer1\', \'denseblock1.denselayer2\', \'norm\']\n        print("Fixing first two conv blocks")\n    elif args.fix_first_conv_block:\n        if \'resne\' in args.encoder:\n            fixing_layers = [\'base_model.conv1\', \'base_model.layer1.0\', \'.bn\']\n        else:\n            fixing_layers = [\'conv0\', \'denseblock1.denselayer1\', \'norm\']\n        print("Fixing first conv block")\n    else:\n        if \'resne\' in args.encoder:\n            fixing_layers = [\'base_model.conv1\', \'.bn\']\n        else:\n            fixing_layers = [\'conv0\', \'norm\']\n        print("Fixing first conv layer")\n\n    for name, child in model.named_children():\n        if not \'encoder\' in name:\n            continue\n        for name2, parameters in child.named_parameters():\n            # print(name, name2)\n            if any(x in name2 for x in fixing_layers):\n                parameters.requires_grad = False\n\n\ndef online_eval(model, dataloader_eval, gpu, ngpus):\n    eval_measures = torch.zeros(10).cuda(device=gpu)\n    for _, eval_sample_batched in enumerate(tqdm(dataloader_eval.data)):\n        with torch.no_grad():\n            image = torch.autograd.Variable(eval_sample_batched[\'image\'].cuda(gpu, non_blocking=True))\n            focal = torch.autograd.Variable(eval_sample_batched[\'focal\'].cuda(gpu, non_blocking=True))\n            gt_depth = eval_sample_batched[\'depth\']\n            has_valid_depth = eval_sample_batched[\'has_valid_depth\']\n            if not has_valid_depth:\n                # print(\'Invalid depth. continue.\')\n                continue\n\n            _, _, _, _, pred_depth = model(image, focal)\n\n            pred_depth = pred_depth.cpu().numpy().squeeze()\n            gt_depth = gt_depth.cpu().numpy().squeeze()\n\n        if args.do_kb_crop:\n            height, width = gt_depth.shape\n            top_margin = int(height - 352)\n            left_margin = int((width - 1216) / 2)\n            pred_depth_uncropped = np.zeros((height, width), dtype=np.float32)\n            pred_depth_uncropped[top_margin:top_margin + 352, left_margin:left_margin + 1216] = pred_depth\n            pred_depth = pred_depth_uncropped\n\n        pred_depth[pred_depth < args.min_depth_eval] = args.min_depth_eval\n        pred_depth[pred_depth > args.max_depth_eval] = args.max_depth_eval\n        pred_depth[np.isinf(pred_depth)] = args.max_depth_eval\n        pred_depth[np.isnan(pred_depth)] = args.min_depth_eval\n\n        valid_mask = np.logical_and(gt_depth > args.min_depth_eval, gt_depth < args.max_depth_eval)\n\n        if args.garg_crop or args.eigen_crop:\n            gt_height, gt_width = gt_depth.shape\n            eval_mask = np.zeros(valid_mask.shape)\n\n            if args.garg_crop:\n                eval_mask[int(0.40810811 * gt_height):int(0.99189189 * gt_height), int(0.03594771 * gt_width):int(0.96405229 * gt_width)] = 1\n\n            elif args.eigen_crop:\n                if args.dataset == \'kitti\':\n                    eval_mask[int(0.3324324 * gt_height):int(0.91351351 * gt_height), int(0.0359477 * gt_width):int(0.96405229 * gt_width)] = 1\n                else:\n                    eval_mask[45:471, 41:601] = 1\n\n            valid_mask = np.logical_and(valid_mask, eval_mask)\n\n        measures = compute_errors(gt_depth[valid_mask], pred_depth[valid_mask])\n\n        eval_measures[:9] += torch.tensor(measures).cuda(device=gpu)\n        eval_measures[9] += 1\n\n    if args.multiprocessing_distributed:\n        group = dist.new_group([i for i in range(ngpus)])\n        dist.all_reduce(tensor=eval_measures, op=dist.ReduceOp.SUM, group=group)\n\n    if not args.multiprocessing_distributed or gpu == 0:\n        eval_measures_cpu = eval_measures.cpu()\n        cnt = eval_measures_cpu[9].item()\n        eval_measures_cpu /= cnt\n        print(\'Computing errors for {} eval samples\'.format(int(cnt)))\n        print("{:>7}, {:>7}, {:>7}, {:>7}, {:>7}, {:>7}, {:>7}, {:>7}, {:>7}".format(\'silog\', \'abs_rel\', \'log10\', \'rms\',\n                                                                                     \'sq_rel\', \'log_rms\', \'d1\', \'d2\',\n                                                                                     \'d3\'))\n        for i in range(8):\n            print(\'{:7.3f}, \'.format(eval_measures_cpu[i]), end=\'\')\n        print(\'{:7.3f}\'.format(eval_measures_cpu[8]))\n        return eval_measures_cpu\n\n    return None\n\n\ndef main_worker(gpu, ngpus_per_node, args):\n    args.gpu = gpu\n\n    if args.gpu is not None:\n        print("Use GPU: {} for training".format(args.gpu))\n\n    if args.distributed:\n        if args.dist_url == "env://" and args.rank == -1:\n            args.rank = int(os.environ["RANK"])\n        if args.multiprocessing_distributed:\n            args.rank = args.rank * ngpus_per_node + gpu\n        dist.init_process_group(backend=args.dist_backend, init_method=args.dist_url, world_size=args.world_size, rank=args.rank)\n\n    # Create model\n    model = BtsModel(args)\n    model.train()\n    model.decoder.apply(weights_init_xavier)\n    set_misc(model)\n\n    num_params = sum([np.prod(p.size()) for p in model.parameters()])\n    print("Total number of parameters: {}".format(num_params))\n\n    num_params_update = sum([np.prod(p.shape) for p in model.parameters() if p.requires_grad])\n    print("Total number of learning parameters: {}".format(num_params_update))\n\n    if args.distributed:\n        if args.gpu is not None:\n            torch.cuda.set_device(args.gpu)\n            model.cuda(args.gpu)\n            args.batch_size = int(args.batch_size / ngpus_per_node)\n            model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[args.gpu], find_unused_parameters=True)\n        else:\n            model.cuda()\n            model = torch.nn.parallel.DistributedDataParallel(model, find_unused_parameters=True)\n    else:\n        model = torch.nn.DataParallel(model)\n        model.cuda()\n\n    if args.distributed:\n        print("Model Initialized on GPU: {}".format(args.gpu))\n    else:\n        print("Model Initialized")\n\n    global_step = 0\n    best_eval_measures_lower_better = torch.zeros(6).cpu() + 1e3\n    best_eval_measures_higher_better = torch.zeros(3).cpu()\n    best_eval_steps = np.zeros(9, dtype=np.int32)\n\n    # Training parameters\n    optimizer = torch.optim.AdamW([{\'params\': model.module.encoder.parameters(), \'weight_decay\': args.weight_decay},\n                                   {\'params\': model.module.decoder.parameters(), \'weight_decay\': 0}],\n                                  lr=args.learning_rate, eps=args.adam_eps)\n\n    model_just_loaded = False\n    if args.checkpoint_path != \'\':\n        if os.path.isfile(args.checkpoint_path):\n            print("Loading checkpoint \'{}\'".format(args.checkpoint_path))\n            if args.gpu is None:\n                checkpoint = torch.load(args.checkpoint_path, weights_only=False)\n            else:\n                loc = \'cuda:{}\'.format(args.gpu)\n                checkpoint = torch.load(args.checkpoint_path, map_location=loc, weights_only=False)\n            last_completed_step = checkpoint[\'global_step\']\n            global_step = last_completed_step + 1\n            model.load_state_dict(checkpoint[\'model\'])\n            optimizer.load_state_dict(checkpoint[\'optimizer\'])\n            try:\n                best_eval_measures_higher_better = checkpoint[\'best_eval_measures_higher_better\'].cpu()\n                best_eval_measures_lower_better = checkpoint[\'best_eval_measures_lower_better\'].cpu()\n                best_eval_steps = checkpoint[\'best_eval_steps\']\n            except KeyError:\n                print("Could not load values for online evaluation")\n\n            print(\n                "Loaded checkpoint \'{}\' "\n                "(last completed global_step {}, "\n                "resuming from global_step {})".format(\n                    args.checkpoint_path,\n                    last_completed_step,\n                    global_step\n                )\n            )\n        else:\n            print("No checkpoint found at \'{}\'".format(args.checkpoint_path))\n        model_just_loaded = True\n\n    if args.retrain:\n        global_step = 0\n\n    cudnn.benchmark = True\n\n    dataloader = BtsDataLoader(args, \'train\')\n    dataloader_eval = BtsDataLoader(args, \'online_eval\')\n\n    # Logging\n    if not args.multiprocessing_distributed or (args.multiprocessing_distributed and args.rank % ngpus_per_node == 0):\n        writer = SummaryWriter(args.log_directory + \'/\' + args.model_name + \'/summaries\', flush_secs=30)\n        if args.do_online_eval:\n            if args.eval_summary_directory != \'\':\n                eval_summary_path = os.path.join(args.eval_summary_directory, args.model_name)\n            else:\n                eval_summary_path = os.path.join(args.log_directory, \'eval\')\n            eval_summary_writer = SummaryWriter(eval_summary_path, flush_secs=30)\n\n        # Initialize Weights & Biases (WandB) on rank 0\n        wandb_active = False\n        try:\n            wandb_key = os.environ.get("WANDB_API_KEY", "")\n            if wandb_key:\n                import wandb\n                wandb.login(key=wandb_key)\n                wandb.init(\n                    project=os.environ.get("WANDB_PROJECT", "bts-nyuv2-depth-research"),\n                    entity=os.environ.get("WANDB_ENTITY", "edward-carlos731-industrial-university-of-ho-chi-minh-city"),\n                    name="bts-densenet161-50ep",\n                    config={\n                        "model": args.model_name,\n                        "encoder": args.encoder,\n                        "dataset": args.dataset,\n                        "epochs": args.num_epochs,\n                        "batch_size": args.batch_size * ngpus_per_node if args.distributed else args.batch_size,\n                        "learning_rate": args.learning_rate,\n                        "weight_decay": args.weight_decay,\n                    },\n                    resume="allow",\n                    id="bts-nyuv2-50ep-run"\n                )\n                wandb_active = True\n                print("Weights & Biases (WandB) initialized successfully!")\n        except Exception as e:\n            print("WandB init skipped/failed:", e)\n            wandb_active = False\n    else:\n        wandb_active = False\n\n    silog_criterion = silog_loss(variance_focus=args.variance_focus)\n\n    start_time = time.time()\n    duration = 0\n\n    num_log_images = args.batch_size\n    end_learning_rate = args.end_learning_rate if args.end_learning_rate != -1 else 0.1 * args.learning_rate\n\n    with torch.no_grad():\n        var_sums = [var.sum() for var in model.parameters() if var.requires_grad]\n        var_cnt = len(var_sums)\n        var_sum = torch.stack(var_sums).sum()\n\n    print(\n        "Initial variables\' sum: {:.3f}, avg: {:.3f}".format(\n            var_sum.item(),\n            var_sum.item() / var_cnt\n        )\n    )\n\n    steps_per_epoch = len(dataloader.data)\n    num_total_steps = args.num_epochs * steps_per_epoch\n    epoch = global_step // steps_per_epoch\n    resume_step_in_epoch = global_step % steps_per_epoch\n\n    while epoch < args.num_epochs:\n        if args.distributed:\n            dataloader.train_sampler.set_epoch(epoch)\n\n        for step, sample_batched in enumerate(dataloader.data):\n            if model_just_loaded and step < resume_step_in_epoch:\n                continue\n\n            if model_just_loaded:\n                print(\n                    "Resume position: epoch {}, "\n                    "step {}/{}, global_step {}".format(\n                        epoch,\n                        step,\n                        steps_per_epoch,\n                        global_step\n                    )\n                )\n                model_just_loaded = False\n\n            optimizer.zero_grad()\n            before_op_time = time.time()\n\n            image = torch.autograd.Variable(sample_batched[\'image\'].cuda(args.gpu, non_blocking=True))\n            focal = torch.autograd.Variable(sample_batched[\'focal\'].cuda(args.gpu, non_blocking=True))\n            depth_gt = torch.autograd.Variable(sample_batched[\'depth\'].cuda(args.gpu, non_blocking=True))\n\n            lpg8x8, lpg4x4, lpg2x2, reduc1x1, depth_est = model(image, focal)\n\n            if args.dataset == \'nyu\':\n                mask = depth_gt > 0.1\n            else:\n                mask = depth_gt > 1.0\n\n            loss = silog_criterion.forward(depth_est, depth_gt, mask.to(torch.bool))\n            loss.backward()\n            for param_group in optimizer.param_groups:\n                current_lr = (args.learning_rate - end_learning_rate) * (1 - global_step / num_total_steps) ** 0.9 + end_learning_rate\n                param_group[\'lr\'] = current_lr\n\n            optimizer.step()\n\n            if not args.multiprocessing_distributed or (args.multiprocessing_distributed and args.rank % ngpus_per_node == 0):\n                print(\'[epoch][s/s_per_e/gs]: [{}][{}/{}/{}], lr: {:.12f}, loss: {:.12f}\'.format(epoch, step, steps_per_epoch, global_step, current_lr, loss))\n                if np.isnan(loss.cpu().item()):\n                    print(\'NaN in loss occurred. Aborting training.\')\n                    return -1\n\n            duration += time.time() - before_op_time\n            if global_step and global_step % args.log_freq == 0 and not model_just_loaded:\n                with torch.no_grad():\n                    var_sums = [var.sum() for var in model.parameters() if var.requires_grad]\n                    var_cnt = len(var_sums)\n                    var_sum = torch.stack(var_sums).sum()\n                examples_per_sec = args.batch_size / duration * args.log_freq\n                duration = 0\n                time_sofar = (time.time() - start_time) / 3600\n                training_time_left = (num_total_steps / global_step - 1.0) * time_sofar\n                if not args.multiprocessing_distributed or (args.multiprocessing_distributed and args.rank % ngpus_per_node == 0):\n                    print("{}".format(args.model_name))\n                print_string = \'GPU: {} | examples/s: {:4.2f} | loss: {:.5f} | var sum: {:.3f} avg: {:.3f} | time elapsed: {:.2f}h | time left: {:.2f}h\'\n                print(print_string.format(args.gpu, examples_per_sec, loss, var_sum.item(), var_sum.item()/var_cnt, time_sofar, training_time_left))\n\n                if not args.multiprocessing_distributed or (args.multiprocessing_distributed\n                                                            and args.rank % ngpus_per_node == 0):\n                    writer.add_scalar(\'silog_loss\', loss, global_step)\n                    writer.add_scalar(\'learning_rate\', current_lr, global_step)\n                    writer.add_scalar(\'var average\', var_sum.item()/var_cnt, global_step)\n                    depth_gt = torch.where(depth_gt < 1e-3, depth_gt * 0 + 1e3, depth_gt)\n                    for i in range(num_log_images):\n                        writer.add_image(\'depth_gt/image/{}\'.format(i), normalize_result(1/depth_gt[i, :, :, :].data), global_step)\n                        writer.add_image(\'depth_est/image/{}\'.format(i), normalize_result(1/depth_est[i, :, :, :].data), global_step)\n                        writer.add_image(\'reduc1x1/image/{}\'.format(i), normalize_result(1/reduc1x1[i, :, :, :].data), global_step)\n                        writer.add_image(\'lpg2x2/image/{}\'.format(i), normalize_result(1/lpg2x2[i, :, :, :].data), global_step)\n                        writer.add_image(\'lpg4x4/image/{}\'.format(i), normalize_result(1/lpg4x4[i, :, :, :].data), global_step)\n                        writer.add_image(\'lpg8x8/image/{}\'.format(i), normalize_result(1/lpg8x8[i, :, :, :].data), global_step)\n                        writer.add_image(\'image/image/{}\'.format(i), inv_normalize(image[i, :, :, :]).data, global_step)\n                    if wandb_active:\n                        try:\n                            import wandb\n                            wandb.log({\n                                "train/silog_loss": loss.item(),\n                                "train/learning_rate": current_lr,\n                                "train/epoch": epoch,\n                                "train/examples_per_sec": examples_per_sec,\n                            }, step=int(global_step))\n                        except Exception:\n                            pass\n                    writer.flush()\n\n            if not args.do_online_eval and global_step and global_step % args.save_freq == 0:\n                if not args.multiprocessing_distributed or (args.multiprocessing_distributed and args.rank % ngpus_per_node == 0):\n                    checkpoint = {\'global_step\': global_step,\n                                  \'model\': model.state_dict(),\n                                  \'optimizer\': optimizer.state_dict()}\n                    torch.save(checkpoint, args.log_directory + \'/\' + args.model_name + \'/model-{}\'.format(global_step))\n\n            if args.do_online_eval and global_step and global_step % args.eval_freq == 0 and not model_just_loaded:\n                time.sleep(0.1)\n                model.eval()\n                eval_measures = online_eval(model, dataloader_eval, gpu, ngpus_per_node)\n                if eval_measures is not None:\n                    for i in range(9):\n                        eval_summary_writer.add_scalar(eval_metrics[i], eval_measures[i].cpu(), int(global_step))\n                        measure = eval_measures[i]\n                        is_best = False\n                        if i < 6 and measure < best_eval_measures_lower_better[i]:\n                            old_best = best_eval_measures_lower_better[i].item()\n                            best_eval_measures_lower_better[i] = measure.item()\n                            is_best = True\n                        elif i >= 6 and measure > best_eval_measures_higher_better[i-6]:\n                            old_best = best_eval_measures_higher_better[i-6].item()\n                            best_eval_measures_higher_better[i-6] = measure.item()\n                            is_best = True\n                        if is_best:\n                            old_best_step = best_eval_steps[i]\n                            old_best_name = \'/model-{}-best_{}_{:.5f}\'.format(old_best_step, eval_metrics[i], old_best)\n                            model_path = args.log_directory + \'/\' + args.model_name + old_best_name\n                            if os.path.exists(model_path):\n                                command = \'rm {}\'.format(model_path)\n                                os.system(command)\n                            best_eval_steps[i] = global_step\n                            model_save_name = \'/model-{}-best_{}_{:.5f}\'.format(global_step, eval_metrics[i], measure)\n                            print(\'New best for {}. Saving model: {}\'.format(eval_metrics[i], model_save_name))\n                            checkpoint = {\'global_step\': global_step,\n                                          \'model\': model.state_dict(),\n                                          \'optimizer\': optimizer.state_dict(),\n                                          \'best_eval_measures_higher_better\': best_eval_measures_higher_better,\n                                          \'best_eval_measures_lower_better\': best_eval_measures_lower_better,\n                                          \'best_eval_steps\': best_eval_steps\n                                          }\n                            torch.save(checkpoint, args.log_directory + \'/\' + args.model_name + model_save_name)\n                    if wandb_active:\n                        try:\n                            import wandb\n                            wandb.log({\n                                "eval/silog": eval_measures[0].item(),\n                                "eval/abs_rel": eval_measures[1].item(),\n                                "eval/log10": eval_measures[2].item(),\n                                "eval/rms": eval_measures[3].item(),\n                                "eval/sq_rel": eval_measures[4].item(),\n                                "eval/log_rms": eval_measures[5].item(),\n                                "eval/delta1": eval_measures[6].item(),\n                                "eval/delta2": eval_measures[7].item(),\n                                "eval/delta3": eval_measures[8].item(),\n                            }, step=int(global_step))\n                        except Exception:\n                            pass\n                    eval_summary_writer.flush()\n                model.train()\n                block_print()\n                set_misc(model)\n                enable_print()\n\n            # Rolling recovery checkpoint.\n            # Keep only one latest checkpoint so long Kaggle runs\n            # can resume without accumulating hundreds of files.\n            if global_step and global_step % args.save_freq == 0:\n                if not args.distributed or args.rank == 0:\n                    checkpoint = {\n                        \'global_step\': global_step,\n                        \'model\': model.state_dict(),\n                        \'optimizer\': optimizer.state_dict(),\n                        \'best_eval_measures_higher_better\':\n                            best_eval_measures_higher_better,\n                        \'best_eval_measures_lower_better\':\n                            best_eval_measures_lower_better,\n                        \'best_eval_steps\':\n                            best_eval_steps\n                    }\n\n                    latest_path = os.path.join(\n                        args.log_directory,\n                        args.model_name,\n                        \'model-latest\'\n                    )\n\n                    temp_path = latest_path + \'.tmp\'\n\n                    torch.save(checkpoint, temp_path)\n                    os.replace(temp_path, latest_path)\n\n                    print(\n                        "Saved recovery checkpoint: {} "\n                        "(global_step {})".format(\n                            latest_path,\n                            global_step\n                        )\n                    )\n\n            model_just_loaded = False\n            global_step += 1\n\n        epoch += 1\n       \n    if not args.multiprocessing_distributed or (args.multiprocessing_distributed and args.rank % ngpus_per_node == 0):\n        writer.close()\n        if args.do_online_eval:\n            eval_summary_writer.close()\n\ndef main():\n    if args.mode != \'train\':\n        print(\'bts_main.py is only for training. Use bts_test.py instead.\')\n        return -1\n\n    model_filename = args.model_name + \'.py\'\n\n    experiment_dir = os.path.join(args.log_directory, args.model_name)\n    os.makedirs(experiment_dir, exist_ok=True)\n\n    args_out_path = os.path.join(\n        experiment_dir,\n        os.path.basename(sys.argv[1])\n    )\n    shutil.copy2(sys.argv[1], args_out_path)\n\n    if args.checkpoint_path == \'\':\n        model_out_path = os.path.join(\n            experiment_dir,\n            model_filename\n        )\n\n        shutil.copy2(\'bts.py\', model_out_path)\n        shutil.copy2(\'bts_main.py\', experiment_dir)\n        shutil.copy2(\'bts_dataloader.py\', experiment_dir)\n\n    else:\n        loaded_model_dir = os.path.dirname(args.checkpoint_path)\n        loaded_model_name = os.path.basename(loaded_model_dir)\n        loaded_model_filename = loaded_model_name + \'.py\'\n\n        model_out_path = os.path.join(\n            experiment_dir,\n            model_filename\n        )\n\n        shutil.copy2(\n            os.path.join(\n                loaded_model_dir,\n                loaded_model_filename\n            ),\n            model_out_path\n        )\n\n    torch.cuda.empty_cache()\n\n    args.distributed = args.world_size > 1 or args.multiprocessing_distributed\n\n    ngpus_per_node = torch.cuda.device_count()\n    if ngpus_per_node > 1 and not args.multiprocessing_distributed:\n        print("This machine has more than 1 gpu. Please specify --multiprocessing_distributed, or set \\\'CUDA_VISIBLE_DEVICES=0\\\'")\n        return -1\n\n    if args.do_online_eval:\n        print("You have specified --do_online_eval.")\n        print("This will evaluate the model every eval_freq {} steps and save best models for individual eval metrics."\n              .format(args.eval_freq))\n\n    if args.multiprocessing_distributed:\n        args.world_size = ngpus_per_node * args.world_size\n        mp.spawn(main_worker, nprocs=ngpus_per_node, args=(ngpus_per_node, args))\n    else:\n        main_worker(args.gpu, ngpus_per_node, args)\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
print('Injected WandB-enabled bts_main.py successfully!')


In [ ]:
weights_src = INPUT / "densenet161-8d451a50.pth"
assert weights_src.exists(), f"Missing pretrained weight: {weights_src}"

weights_hash = sha256_file(weights_src)
assert weights_hash.startswith("8d451a50")

cache_dir = Path(torch.hub.get_dir()) / "checkpoints"
cache_dir.mkdir(parents=True, exist_ok=True)

weights_dst = cache_dir / weights_src.name
shutil.copy2(weights_src, weights_dst)

print("DenseNet161:", weights_dst)
print("SHA256:", weights_hash)
print("Size:", round(weights_dst.stat().st_size / 1024**2, 2), "MB")


In [ ]:
import importlib.util
import subprocess

if importlib.util.find_spec("tensorboardX") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "tensorboardX"]
    )

import numpy as np
import pandas as pd
import cv2
import scipy
import h5py
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("SciPy:", scipy.__version__)
print("h5py:", h5py.__version__)
print("tensorboardX: available")


## 5. Kiểm tra tính toàn vẹn dữ liệu

Không bắt đầu training nếu file list và dữ liệu vật lý không khớp. Toàn bộ đường dẫn RGB và depth ground truth của train/test được kiểm tra trước khi khởi tạo model.


In [ ]:
train_list = BTS_ROOT / "train_test_inputs" / "nyudepthv2_train_files_with_gt.txt"
test_list = BTS_ROOT / "train_test_inputs" / "nyudepthv2_test_files_with_gt.txt"

train_rows = [
    line.strip().split()
    for line in train_list.read_text().splitlines()
    if line.strip()
]
test_rows = [
    line.strip().split()
    for line in test_list.read_text().splitlines()
    if line.strip()
]

missing_train_rgb = [
    row[0] for row in train_rows
    if not (train_root / row[0].lstrip("/")).exists()
]
missing_train_gt = [
    row[1] for row in train_rows
    if not (train_root / row[1].lstrip("/")).exists()
]
missing_test_rgb = [
    row[0] for row in test_rows
    if not (test_root / row[0].lstrip("/")).exists()
]
missing_test_gt = [
    row[1] for row in test_rows
    if not (test_root / row[1].lstrip("/")).exists()
]

integrity = pd.DataFrame([
    {
        "Split": "Train",
        "Samples": len(train_rows),
        "Missing RGB": len(missing_train_rgb),
        "Missing depth": len(missing_train_gt),
    },
    {
        "Split": "Test",
        "Samples": len(test_rows),
        "Missing RGB": len(missing_test_rgb),
        "Missing depth": len(missing_test_gt),
    },
])

display(integrity)

assert len(train_rows) == 24231
assert len(test_rows) == 654
assert not missing_train_rgb
assert not missing_train_gt
assert not missing_test_rgb
assert not missing_test_gt

print("Dataset integrity check completed.")


## 6. Tương thích runtime và checkpoint

Mã nguồn BTS được giữ nguyên về kiến trúc, loss, optimizer và protocol đánh giá. Runtime chỉ bổ sung các cơ chế kỹ thuật cần thiết:

- checkpoint loading tương thích PyTorch hiện đại;
- rolling `model-latest` để phục hồi khi Kaggle session bị ngắt;
- resume đúng vị trí giữa epoch;
- `model-final` lưu đúng trạng thái sau optimization step cuối.

Các thay đổi này không thay đổi phép tính forward, SILog loss, AdamW update hay metric đánh giá.


In [ ]:
import re
import subprocess

bts_main_path = BTS_ROOT / "pytorch" / "bts_main.py"
bts_test_path = BTS_ROOT / "pytorch" / "bts_test.py"

assert bts_main_path.exists()
assert bts_test_path.exists()

def ensure_weights_only_false(path: Path):
    text = path.read_text(encoding="utf-8")
    replacements = [
        (
            "torch.load(args.checkpoint_path, map_location=loc)",
            "torch.load(args.checkpoint_path, map_location=loc, weights_only=False)",
        ),
        (
            "torch.load(args.checkpoint_path)",
            "torch.load(args.checkpoint_path, weights_only=False)",
        ),
    ]
    for old, new in replacements:
        if new not in text and old in text:
            text = text.replace(old, new)
    path.write_text(text, encoding="utf-8")

def ensure_mid_epoch_resume(path: Path):
    text = path.read_text(encoding="utf-8")

    markers = [
        "last_completed_step = checkpoint['global_step']",
        "global_step = last_completed_step + 1",
        "resume_step_in_epoch = global_step % steps_per_epoch",
        "if model_just_loaded and step < resume_step_in_epoch:",
    ]
    if all(marker in text for marker in markers):
        return

    old_load = (
        "            global_step = checkpoint['global_step']\n"
        "            model.load_state_dict(checkpoint['model'])"
    )
    new_load = (
        "            last_completed_step = checkpoint['global_step']\n"
        "            global_step = last_completed_step + 1\n"
        "            model.load_state_dict(checkpoint['model'])"
    )
    if old_load not in text:
        raise RuntimeError("Không tìm thấy checkpoint loading block.")
    text = text.replace(old_load, new_load, 1)

    old_print = (
        "            print(\"Loaded checkpoint '{}' (global_step {})\".format("
        "args.checkpoint_path, checkpoint['global_step']))"
    )
    new_print = """            print(
                "Loaded checkpoint '{}' "
                "(last completed global_step {}, "
                "resuming from global_step {})".format(
                    args.checkpoint_path,
                    last_completed_step,
                    global_step
                )
            )"""
    if old_print in text:
        text = text.replace(old_print, new_print, 1)

    epoch_line = "    epoch = global_step // steps_per_epoch"
    if "resume_step_in_epoch = global_step % steps_per_epoch" not in text:
        text = text.replace(
            epoch_line,
            epoch_line + "\n    resume_step_in_epoch = global_step % steps_per_epoch",
            1,
        )

    old_loop = (
        "        for step, sample_batched in enumerate(dataloader.data):\n"
        "            optimizer.zero_grad()"
    )
    new_loop = """        for step, sample_batched in enumerate(dataloader.data):
            if model_just_loaded and step < resume_step_in_epoch:
                continue

            if model_just_loaded:
                print(
                    "Resume position: epoch {}, "
                    "step {}/{}, global_step {}".format(
                        epoch,
                        step,
                        steps_per_epoch,
                        global_step
                    )
                )
                model_just_loaded = False

            optimizer.zero_grad()"""
    if "if model_just_loaded and step < resume_step_in_epoch:" not in text:
        if old_loop not in text:
            raise RuntimeError("Không tìm thấy training-loop insertion point.")
        text = text.replace(old_loop, new_loop, 1)

    path.write_text(text, encoding="utf-8")

FINAL_PATCH_MARKER = "# FINAL_CHECKPOINT_PATCH_V1"

def ensure_final_checkpoint(path: Path):
    text = path.read_text(encoding="utf-8")
    if FINAL_PATCH_MARKER in text:
        return

    anchor = (
        "    if not args.multiprocessing_distributed or "
        "(args.multiprocessing_distributed "
        "and args.rank % ngpus_per_node == 0):\n"
        "        writer.close()"
    )
    if anchor not in text:
        raise RuntimeError("Không tìm thấy writer-close anchor.")

    block = """    # FINAL_CHECKPOINT_PATCH_V1
    if global_step > 0 and (not args.distributed or args.rank == 0):
        final_completed_step = global_step - 1

        final_checkpoint = {
            'global_step': final_completed_step,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'best_eval_measures_higher_better': best_eval_measures_higher_better,
            'best_eval_measures_lower_better': best_eval_measures_lower_better,
            'best_eval_steps': best_eval_steps
        }

        experiment_dir = os.path.join(
            args.log_directory,
            args.model_name
        )

        final_path = os.path.join(experiment_dir, 'model-final')
        final_tmp = final_path + '.tmp'
        torch.save(final_checkpoint, final_tmp)
        os.replace(final_tmp, final_path)

        latest_path = os.path.join(experiment_dir, 'model-latest')
        latest_tmp = latest_path + '.tmp'
        shutil.copy2(final_path, latest_tmp)
        os.replace(latest_tmp, latest_path)

        print(
            "Saved final checkpoint: {} "
            "(global_step {})".format(
                final_path,
                final_completed_step
            )
        )

"""
    text = text.replace(anchor, block + anchor, 1)
    path.write_text(text, encoding="utf-8")

source_text = bts_main_path.read_text(encoding="utf-8")
assert "model-latest" in source_text
assert "os.replace" in source_text
assert "Saved recovery checkpoint" in source_text

ensure_mid_epoch_resume(bts_main_path)
ensure_final_checkpoint(bts_main_path)
ensure_weights_only_false(bts_main_path)
ensure_weights_only_false(bts_test_path)

for path in [bts_main_path, bts_test_path]:
    result = subprocess.run(
        [sys.executable, "-m", "py_compile", str(path)],
        capture_output=True,
        text=True,
    )
    assert result.returncode == 0, result.stderr

runtime_checks = pd.DataFrame([
    {"Check": "Rolling recovery checkpoint", "Status": "OK"},
    {"Check": "Mid-epoch resume", "Status": "OK"},
    {"Check": "Final checkpoint", "Status": "OK"},
    {"Check": "PyTorch checkpoint loading", "Status": "OK"},
])

display(runtime_checks)
print("Runtime BTS SHA256:", sha256_file(bts_main_path))


## 7. Cấu hình huấn luyện đầy đủ

Với 24,231 mẫu và 2 GPU, `DistributedSampler` phân phối 12,116 mẫu cho mỗi GPU. Batch cục bộ là 2, tương ứng **6,058 optimization steps/epoch** và **302,900 steps cho 50 epochs**.

Learning rate được giữ theo schedule của BTS và online evaluation được thực hiện mỗi 500 global steps.


In [ ]:
import math
import gc
import json

PYTORCH_DIR = BTS_ROOT / "pytorch"
MODELS_DIR = WORK / "models"
LOGS_DIR = WORK / "logs"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

def write_config(filename: str, text: str) -> Path:
    path = PYTORCH_DIR / filename
    path.write_text(text.strip() + "\n", encoding="utf-8")
    return path

def checkpoint_summary(path: Path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    best_steps = ckpt.get("best_eval_steps")
    summary = {
        "path": str(path),
        "size_mb": round(path.stat().st_size / 1024**2, 2),
        "global_step": int(ckpt["global_step"]),
        "model_tensors": len(ckpt["model"]),
        "optimizer_groups": len(ckpt["optimizer"]["param_groups"]),
        "best_eval_steps": (
            best_steps.tolist()
            if hasattr(best_steps, "tolist")
            else best_steps
        ),
    }
    del ckpt
    gc.collect()
    return summary

def stage_resume_checkpoint(source: Path) -> Path:
    source = Path(source)
    assert source.exists(), source

    stage_dir = TEMP / "resume" / MODEL_NAME
    if stage_dir.exists():
        shutil.rmtree(stage_dir)
    stage_dir.mkdir(parents=True, exist_ok=True)

    staged_ckpt = stage_dir / "model-latest"
    shutil.copy2(source, staged_ckpt)

    candidates = [
        source.parent / f"{source.parent.name}.py",
        source.parent / f"{MODEL_NAME}.py",
    ]
    architecture = next((p for p in candidates if p.exists()), None)
    if architecture is None:
        architecture = BTS_ROOT / "pytorch" / "bts.py"

    shutil.copy2(
        architecture,
        stage_dir / f"{MODEL_NAME}.py",
    )
    return staged_ckpt

resume_staged = None

if RESUME_CHECKPOINT.strip():
    resume_staged = stage_resume_checkpoint(
        Path(RESUME_CHECKPOINT.strip())
    )
else:
    working_latest = MODELS_DIR / MODEL_NAME / "model-latest"
    if working_latest.exists():
        resume_staged = stage_resume_checkpoint(working_latest)

if resume_staged:
    print("Training mode: resume")
    display(pd.DataFrame([checkpoint_summary(resume_staged)]))
else:
    print("Training mode: fresh")


In [ ]:
samples_per_gpu = math.ceil(len(train_rows) / 2)
local_batch_size = GLOBAL_BATCH_SIZE // 2
steps_per_epoch = math.ceil(samples_per_gpu / local_batch_size)
total_steps = steps_per_epoch * TRAIN_EPOCHS
expected_final_step = total_steps - 1

resume_line = (
    f"--checkpoint_path {resume_staged}"
    if resume_staged is not None
    else ""
)

full_config = f"""
--mode train
--model_name {MODEL_NAME}
--encoder densenet161_bts
--dataset nyu

--data_path {train_root}/
--gt_path {train_root}/
--filenames_file {train_list}

--batch_size {GLOBAL_BATCH_SIZE}
--num_epochs {TRAIN_EPOCHS}
--learning_rate {LEARNING_RATE}
--weight_decay {WEIGHT_DECAY}
--adam_eps {ADAM_EPS}
--num_threads 1

--input_height {INPUT_HEIGHT}
--input_width {INPUT_WIDTH}
--max_depth {MAX_DEPTH}

--do_random_rotate
--degree 2.5

--log_directory {MODELS_DIR}/
--log_freq 100
--save_freq {SAVE_FREQ}

--multiprocessing_distributed
--dist_url tcp://127.0.0.1:2350

--do_online_eval
--eval_freq {EVAL_FREQ}

--data_path_eval {test_root}/
--gt_path_eval {test_root}/
--filenames_file_eval {test_list}
--min_depth_eval 1e-3
--max_depth_eval 10
--eval_summary_directory {MODELS_DIR}/eval/
--eigen_crop

{resume_line}
"""

full_cfg = write_config(
    "arguments_train_nyu_kaggle_full.txt",
    full_config,
)

protocol = pd.DataFrame([
    {"Parameter": "Train samples", "Value": len(train_rows)},
    {"Parameter": "Test samples", "Value": len(test_rows)},
    {"Parameter": "Global batch size", "Value": GLOBAL_BATCH_SIZE},
    {"Parameter": "Local batch / GPU", "Value": local_batch_size},
    {"Parameter": "Steps / epoch", "Value": steps_per_epoch},
    {"Parameter": "Epochs", "Value": TRAIN_EPOCHS},
    {"Parameter": "Total optimization steps", "Value": total_steps},
    {"Parameter": "Expected final global_step", "Value": expected_final_step},
])

display(protocol)
print("Training config:", full_cfg)


## 8. Huấn luyện

Toàn bộ stdout/stderr được lưu tại:

`/kaggle/working/logs/bts_full_train.log`

Notebook chỉ hiển thị các mốc training/evaluation quan trọng để tránh tạo output quá lớn. `model-latest` được cập nhật định kỳ và dùng làm recovery checkpoint nếu session bị gián đoạn.


In [ ]:
progress_pattern = re.compile(
    r"\[(\d+)\]\[(\d+)/(\d+)/(\d+)\], "
    r"lr:\s*([0-9.eE+-]+), loss:\s*([0-9.eE+-]+)"
)

def run_training(config_path: Path, log_path: Path, display_every=500):
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    cmd = [
        sys.executable,
        "-u",
        "bts_main.py",
        config_path.name,
    ]

    print("Command:", " ".join(cmd))
    print("Log:", log_path)

    process = subprocess.Popen(
        cmd,
        cwd=PYTORCH_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    show_next_eval_line = 0

    with log_path.open("w", encoding="utf-8", buffering=1) as log:
        for line in process.stdout:
            log.write(line)

            match = progress_pattern.search(line)
            if match:
                step_in_epoch = int(match.group(2))
                total_in_epoch = int(match.group(3))
                if (
                    step_in_epoch % display_every == 0
                    or step_in_epoch == total_in_epoch - 1
                ):
                    print(line, end="")
                continue

            if "Computing errors for" in line:
                print(line, end="")
                show_next_eval_line = 2
                continue

            if show_next_eval_line > 0:
                print(line, end="")
                show_next_eval_line -= 1
                continue

            important = (
                "Use GPU:",
                "Model Initialized",
                "Loading checkpoint",
                "Loaded checkpoint",
                "Resume position:",
                "New best for",
                "Saved recovery checkpoint:",
                "Saved final checkpoint:",
                "NaN in loss",
                "Traceback",
                "RuntimeError",
            )
            if any(token in line for token in important):
                print(line, end="")

    returncode = process.wait()
    print("Return code:", returncode)

    if returncode != 0:
        tail = log_path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()[-100:]
        print("\n".join(tail))
        raise RuntimeError(
            f"Training failed with return code {returncode}"
        )

    return log_path


In [ ]:
FULL_MODEL_DIR = MODELS_DIR / MODEL_NAME
FULL_TRAIN_LOG = LOGS_DIR / "bts_full_train.log"

if RUN_TRAINING:
    if resume_staged is None and FULL_MODEL_DIR.exists():
        print("Removing stale fresh-run directory:", FULL_MODEL_DIR)
        shutil.rmtree(FULL_MODEL_DIR)

    run_training(
        full_cfg,
        FULL_TRAIN_LOG,
        display_every=500,
    )

    model_final = FULL_MODEL_DIR / "model-final"
    assert model_final.exists(), (
        "Training completed but model-final was not created."
    )

    final_checkpoint_info = checkpoint_summary(model_final)
    display(pd.DataFrame([final_checkpoint_info]))

    assert final_checkpoint_info["global_step"] == expected_final_step
    print("Full 50-epoch training completed.")
else:
    print("Training skipped.")


## 9. Diễn biến huấn luyện

Hai đồ thị dưới đây được sinh trực tiếp từ log của full run:

- SILog training loss theo global step;
- learning-rate schedule theo global step.

Đường trung bình trượt của loss được dùng để quan sát xu hướng tổng thể vì loss theo mini-batch có dao động tự nhiên.


In [ ]:
def parse_training_log(log_path: Path) -> pd.DataFrame:
    rows = []
    if not log_path.exists():
        return pd.DataFrame()

    for line in log_path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines():
        m = progress_pattern.search(line)
        if not m:
            continue

        rows.append({
            "epoch": int(m.group(1)),
            "step_in_epoch": int(m.group(2)),
            "steps_per_epoch": int(m.group(3)),
            "global_step": int(m.group(4)),
            "learning_rate": float(m.group(5)),
            "loss": float(m.group(6)),
        })

    return pd.DataFrame(rows)

history = parse_training_log(FULL_TRAIN_LOG)

if history.empty:
    print("Training history is not available yet.")
else:
    history.to_csv(
        WORK / "bts_nyuv2_training_history.csv",
        index=False,
    )

    history["loss_rolling_500"] = (
        history["loss"]
        .rolling(500, min_periods=1)
        .mean()
    )

    display(history.tail())

    plt.figure(figsize=(10, 5))
    plt.plot(history["global_step"], history["loss"], alpha=0.2, label="Batch loss")
    plt.plot(
        history["global_step"],
        history["loss_rolling_500"],
        label="Rolling mean (500 steps)",
    )
    plt.xlabel("Global step")
    plt.ylabel("SILog loss")
    plt.title("BTS training loss on NYU Depth V2")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(history["global_step"], history["learning_rate"])
    plt.xlabel("Global step")
    plt.ylabel("Learning rate")
    plt.title("Learning-rate schedule")
    plt.grid(alpha=0.2)
    plt.show()


## 10. Suy luận và đánh giá cuối cùng

Sau khi full training hoàn tất, `model-final` được dùng để dự đoán depth cho toàn bộ 654 ảnh test. Metric được tính bằng script đánh giá của BTS với NYUv2 Eigen crop.

Các metric báo cáo:

- δ1, δ2, δ3;
- AbsRel;
- SqRel;
- RMSE;
- RMSElog;
- SILog;
- log10.


In [ ]:
FINAL_CHECKPOINT = FULL_MODEL_DIR / "model-final"
FINAL_RESULT_DIR = WORK / f"result_{MODEL_NAME}"
FINAL_TEST_LOG = LOGS_DIR / "bts_final_test.log"
FINAL_EVAL_LOG = LOGS_DIR / "bts_final_eval.log"

test_config = f"""
--encoder densenet161_bts
--data_path {test_root}/
--dataset nyu
--filenames_file {test_list}
--model_name {MODEL_NAME}
--checkpoint_path {FINAL_CHECKPOINT}
--input_height 480
--input_width 640
--max_depth 10
"""

final_test_cfg = write_config(
    "arguments_test_nyu_kaggle_final.txt",
    test_config,
)

if RUN_FINAL_EVALUATION:
    assert FINAL_CHECKPOINT.exists(), (
        "model-final chưa tồn tại. Hoàn tất training trước."
    )

    if FINAL_RESULT_DIR.exists():
        shutil.rmtree(FINAL_RESULT_DIR)

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    with FINAL_TEST_LOG.open("w", encoding="utf-8") as log:
        result = subprocess.run(
            [
                sys.executable,
                "-u",
                str(PYTORCH_DIR / "bts_test.py"),
                str(final_test_cfg),
            ],
            cwd=WORK,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
            env=env,
        )

    if result.returncode != 0:
        print(FINAL_TEST_LOG.read_text(
            encoding="utf-8",
            errors="replace",
        )[-8000:])
    assert result.returncode == 0

    raw_dir = FINAL_RESULT_DIR / "raw"
    raw_count = len(list(raw_dir.glob("*.png")))

    print("Prediction PNGs:", raw_count)
    assert raw_count == 654

    eval_result = subprocess.run(
        [
            sys.executable,
            str(BTS_ROOT / "utils" / "eval_with_pngs.py"),
            "--pred_path", str(raw_dir),
            "--gt_path", str(test_root),
            "--dataset", "nyu",
            "--eigen_crop",
            "--min_depth_eval", "1e-3",
            "--max_depth_eval", "10",
        ],
        cwd=WORK,
        capture_output=True,
        text=True,
    )

    FINAL_EVAL_LOG.write_text(
        eval_result.stdout + "\n" + eval_result.stderr,
        encoding="utf-8",
    )

    print(eval_result.stdout)
    assert eval_result.returncode == 0
    assert "Evaluating 654 files" in eval_result.stdout

    print("Final evaluation completed.")
else:
    print("Final evaluation skipped.")


## 11. Kết quả định lượng

Bảng được tạo trực tiếp từ output của `eval_with_pngs.py`.

Hàng **Author checkpoint reproduction** là kết quả sanity-check đã chạy trước đó với checkpoint phát hành bởi tác giả. Hàng **Our 50-epoch run** là kết quả của model được huấn luyện trong notebook này.


In [ ]:
METRIC_NAMES = [
    "d1", "d2", "d3",
    "AbsRel", "SqRel", "RMSE",
    "RMSElog", "SILog", "log10",
]

REFERENCE_METRICS = {
    "d1": 0.885,
    "d2": 0.978,
    "d3": 0.994,
    "AbsRel": 0.110,
    "SqRel": 0.066,
    "RMSE": 0.392,
    "RMSElog": 0.142,
    "SILog": 11.535,
    "log10": 0.047,
}

def parse_final_metrics(log_path: Path):
    if not log_path.exists():
        return None

    lines = log_path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    for i, line in enumerate(lines):
        if (
            "d1" in line
            and "AbsRel" in line
            and "SILog" in line
            and i + 1 < len(lines)
        ):
            values = [
                float(x.strip())
                for x in lines[i + 1].split(",")
            ]
            if len(values) == 9:
                return dict(zip(METRIC_NAMES, values))

    return None

final_metrics = parse_final_metrics(FINAL_EVAL_LOG)

comparison_rows = [
    {
        "Experiment": "Author checkpoint reproduction",
        **REFERENCE_METRICS,
    }
]

if final_metrics is not None:
    comparison_rows.append({
        "Experiment": "Our 50-epoch run",
        **final_metrics,
    })

comparison = pd.DataFrame(comparison_rows)
display(comparison)

comparison.to_csv(
    WORK / "bts_nyuv2_final_metrics.csv",
    index=False,
)

if final_metrics is None:
    print("Final metrics will appear after final evaluation.")


## 12. Ví dụ định tính

Các ví dụ dưới đây lấy trực tiếp từ official test split. Mỗi mẫu gồm ảnh RGB đầu vào, depth ground truth và depth dự đoán của `model-final`.


In [ ]:
def prediction_name_from_row(row):
    rgb_rel = row[0].lstrip("/")
    parts = rgb_rel.split("/")
    scene = parts[0]
    rgb_name = Path(parts[-1]).with_suffix(".png").name
    return f"{scene}_{rgb_name}"

def show_depth_example(index: int):
    row = test_rows[index]

    rgb_path = test_root / row[0].lstrip("/")
    gt_path = test_root / row[1].lstrip("/")
    pred_path = FINAL_RESULT_DIR / "raw" / prediction_name_from_row(row)

    assert rgb_path.exists()
    assert gt_path.exists()
    assert pred_path.exists()

    rgb = cv2.cvtColor(
        cv2.imread(str(rgb_path)),
        cv2.COLOR_BGR2RGB,
    )
    gt = cv2.imread(str(gt_path), -1).astype(np.float32) / 1000.0
    pred = cv2.imread(str(pred_path), -1).astype(np.float32) / 1000.0

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].imshow(rgb)
    axes[0].set_title("RGB")
    axes[0].axis("off")

    axes[1].imshow(gt, vmin=0, vmax=MAX_DEPTH)
    axes[1].set_title("Ground truth depth (m)")
    axes[1].axis("off")

    axes[2].imshow(pred, vmin=0, vmax=MAX_DEPTH)
    axes[2].set_title("Predicted depth (m)")
    axes[2].axis("off")

    fig.suptitle(f"NYUv2 test sample #{index}")
    plt.tight_layout()
    plt.show()

if RUN_FINAL_EVALUATION and FINAL_RESULT_DIR.exists():
    for idx in [0, len(test_rows) // 2, len(test_rows) - 1]:
        show_depth_example(idx)
else:
    print("Qualitative examples will appear after final evaluation.")


## 13. Tổng hợp thí nghiệm và lưu thông tin tái lập

Manifest cuối cùng lưu môi trường phần mềm, GPU, kích thước dataset, SHA256 của source/runtime, training protocol, checkpoint và metric cuối.


In [ ]:
import datetime
import platform

manifest = {
    "created_utc": datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat(),
    "experiment": "BTS NYUv2 full baseline",
    "model": "BTS",
    "encoder": "DenseNet161",
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu_count": torch.cuda.device_count(),
    "gpus": [
        torch.cuda.get_device_name(i)
        for i in range(torch.cuda.device_count())
    ],
    "train_samples": len(train_rows),
    "test_samples": len(test_rows),
    "input_size": [INPUT_HEIGHT, INPUT_WIDTH],
    "global_batch_size": GLOBAL_BATCH_SIZE,
    "epochs": TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "adam_eps": ADAM_EPS,
    "max_depth": MAX_DEPTH,
    "eval_freq": EVAL_FREQ,
    "save_freq": SAVE_FREQ,
    "steps_per_epoch": steps_per_epoch,
    "total_steps": total_steps,
    "expected_final_global_step": expected_final_step,
    "bts_main_sha256_runtime": sha256_file(bts_main_path),
    "densenet161_sha256": weights_hash,
    "resume_checkpoint_requested": RESUME_CHECKPOINT,
}

model_latest = FULL_MODEL_DIR / "model-latest"
model_final = FULL_MODEL_DIR / "model-final"

if model_latest.exists():
    manifest["model_latest"] = checkpoint_summary(model_latest)

if model_final.exists():
    manifest["model_final"] = checkpoint_summary(model_final)

if final_metrics is not None:
    manifest["final_metrics"] = final_metrics

manifest_path = WORK / "bts_nyuv2_experiment_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("Manifest:", manifest_path)
display(pd.DataFrame([
    {"Artifact": "Training log", "Path": str(FULL_TRAIN_LOG)},
    {"Artifact": "Training history", "Path": str(WORK / "bts_nyuv2_training_history.csv")},
    {"Artifact": "Final metrics", "Path": str(WORK / "bts_nyuv2_final_metrics.csv")},
    {"Artifact": "Manifest", "Path": str(manifest_path)},
    {"Artifact": "Final checkpoint", "Path": str(model_final)},
]))


## 14. Kết luận baseline

Baseline BTS được xem là hoàn thành khi thỏa đồng thời:

- full 50 epochs kết thúc;
- `model-final` có `global_step = 302899`;
- 654 prediction PNG được sinh;
- final evaluation chạy đủ 654 mẫu;
- bảng 9 metric được tạo;
- training history, log và manifest được lưu.

Baseline này là mốc tham chiếu cho các thí nghiệm tiếp theo. Mọi cải tiến mô hình nên giữ cố định split dữ liệu, depth range, evaluation crop và metric để bảo đảm so sánh công bằng.

### Tài liệu tham khảo

1. Jin Han Lee et al., *From Big to Small: Multi-Scale Local Planar Guidance for Monocular Depth Estimation*.
2. BTS official repository: `https://github.com/cleinc/bts`
3. NYU Depth V2 dataset.
